# CamDocLM — LayoutLMv3 fine-tuning (Cameroon NIC v1)

Trains a token-classification model to extract fields from Cameroon NIC v1 cards.

**Before running:** upload/extract your `data/output/Cameroon_NIC_v1/` and `hf_data/nic_v1/` folders here, preserving the same relative paths used when you ran `convert_to_hf_dataset.py` — the JSONL's `image_path` values are relative to your project root, so Colab's working directory needs to mirror that structure (e.g. zip the whole project folder, upload, and `!unzip`).

**Deployment reminder:** this model expects *words + boxes* as input, not a raw photo. A real deployment pipeline is OCR → this model → BIO-decode (see the last section for the decode step; OCR integration is a separate next step once training checks out).

In [ ]:
!pip install -q transformers datasets seqeval accelerate

## 1. Load labels and data

In [ ]:
import json
from datasets import load_dataset

DATA_DIR = "hf_data/nic_v1"

with open(f"{DATA_DIR}/labels.json") as f:
    label_map = json.load(f)

id2label = {int(k): v for k, v in label_map["id2label"].items()}
label2id = label_map["label2id"]
print(f"{len(id2label)} labels:", list(id2label.values()))

raw_datasets = load_dataset("json", data_files={
    "train": f"{DATA_DIR}/train.jsonl",
    "validation": f"{DATA_DIR}/val.jsonl",
    "test": f"{DATA_DIR}/test.jsonl",
})
raw_datasets

## 2. Preprocess with the LayoutLMv3 processor
`apply_ocr=False` because we already have words/boxes — the processor just tokenizes and aligns them to the image.

In [ ]:
from transformers import LayoutLMv3Processor
from PIL import Image

processor = LayoutLMv3Processor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=False)

def prepare_examples(examples):
    images = [Image.open(p).convert("RGB") for p in examples["image_path"]]
    words = examples["words"]
    boxes = examples["bboxes"]
    word_labels = [[label2id[tag] for tag in tags] for tags in examples["ner_tags"]]
    encoding = processor(
        images, words, boxes=boxes, word_labels=word_labels,
        truncation=True, padding="max_length",
    )
    return encoding

column_names = raw_datasets["train"].column_names
train_dataset = raw_datasets["train"].map(prepare_examples, batched=True, remove_columns=column_names)
val_dataset = raw_datasets["validation"].map(prepare_examples, batched=True, remove_columns=column_names)
test_dataset = raw_datasets["test"].map(prepare_examples, batched=True, remove_columns=column_names)

train_dataset.set_format("torch")
val_dataset.set_format("torch")
test_dataset.set_format("torch")

## 3. Load the model

In [ ]:
from transformers import LayoutLMv3ForTokenClassification

model = LayoutLMv3ForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base",
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id,
)

## 4. Metrics (entity-level F1 via seqeval)

In [ ]:
import numpy as np
from seqeval.metrics import precision_score, recall_score, f1_score

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id2label[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
    }

## 5. Train
Starting point for a first run — watch validation F1 and adjust epochs/learning rate if it's still climbing or has plateaued.

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./layoutlmv3-nic-v1",
    num_train_epochs=15,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=1e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=20,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

## 6. Evaluate on the held-out test set

In [ ]:
test_results = trainer.evaluate(test_dataset)
test_results

## 7. Save the model (for deployment)

In [ ]:
trainer.save_model("./layoutlmv3-nic-v1-final")
processor.save_pretrained("./layoutlmv3-nic-v1-final")

## 8. Demo: decode predictions back into field values
Uses the known words/boxes from a test example (not real OCR — see the deployment reminder at the top). This confirms the BIO-decode logic works before wiring up a real OCR front-end.

In [ ]:
import torch

def decode_predictions(words, predicted_tags):
    fields = {}
    current_label, current_words = None, []
    for word, tag in list(zip(words, predicted_tags)) + [(None, "O")]:
        if tag.startswith("B-"):
            if current_label:
                fields[current_label] = " ".join(current_words)
            current_label, current_words = tag[2:], [word]
        elif tag.startswith("I-") and current_label == tag[2:]:
            current_words.append(word)
        else:
            if current_label:
                fields[current_label] = " ".join(current_words)
            current_label, current_words = None, []
    return fields

example = raw_datasets["test"][0]
image = Image.open(example["image_path"]).convert("RGB")
encoding = processor(image, example["words"], boxes=example["bboxes"],
                      truncation=True, padding="max_length", return_tensors="pt")

with torch.no_grad():
    outputs = model(**encoding)

predicted_ids = outputs.logits.argmax(-1).squeeze().tolist()
predicted_tags = [id2label[i] for i in predicted_ids]

# word_ids() maps each token back to its original word, since the
# tokenizer may split one word into multiple subword tokens
word_ids = encoding.word_ids(0)
seen = set()
aligned_words, aligned_tags = [], []
for idx, wid in enumerate(word_ids):
    if wid is None or wid in seen:
        continue
    seen.add(wid)
    aligned_words.append(example["words"][wid])
    aligned_tags.append(predicted_tags[idx])

print("Ground truth words:", example["words"])
print("Predicted fields:", decode_predictions(aligned_words, aligned_tags))